# 10 — Persistence：会话持久化与检查点

Agent 任务可能跑很久，用户不可能一直等待。这章实现两个核心能力：

1. **会话持久化（Session Persistence）**：把对话历史保存到磁盘，下次启动可以继续上次的任务
2. **检查点（Checkpoint）**：在某个时间点保存快照，出错或走偏时可以回退

```
时间线：

T0: 开始对话 ──┬── /checkpoint → ck_001
               │
T1: 继续对话 ──┤── /checkpoint → ck_002
               │
T2: 出错了    ──┤
               │
T3: /restore ck_001 → 回到 T0 的状态
```

## 文件布局

```
.ai_agent/
├── sessions/
│   └── {session_id}.json          ← 会话当前状态
└── checkpoints/
    └── {session_id}_{YYYYMMDD_HHMMSS}.json  ← 某个时间点的快照
```

会话文件随每次 `/save` 覆盖更新；检查点文件只追加，不修改。

In [ ]:
import sys
import os
sys.path.insert(0, "..")

# 检查依赖
try:
    import aiofiles
    print("aiofiles 已安装")
except ImportError:
    print("安装 aiofiles...")
    os.system("pip install aiofiles -q")
    import aiofiles
    print("aiofiles 安装完成")

## 1. SessionSnapshot 数据类

`SessionSnapshot` 是序列化的单元——它把 `Session` 当前状态打包成一个可以存到磁盘、也可以从磁盘还原的对象。

```
Session (内存中的活跃对象)
    │
    ▼  to_dict() / from_dict()
SessionSnapshot (可序列化的快照)
    │
    ▼  json.dumps / json.loads
磁盘 JSON 文件
```

关键字段：
- `messages`：`ctx.get_messages()` 的结果，包含完整的对话历史
- `total_usage`：累计 token 消耗
- `turn_count`：已经进行了多少轮对话

In [ ]:
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional
import json


@dataclass
class SessionSnapshot:
    """会话状态的完整快照，可序列化为 JSON。"""

    session_id: str
    created_at: str          # ISO 8601 格式，带时区
    updated_at: str
    turn_count: int
    messages: list           # ctx.get_messages() 的原始结果
    total_usage: dict        # {"prompt_tokens": int, "completion_tokens": int, "total_tokens": int}
    system_prompt: str = ""
    metadata: dict = field(default_factory=dict)

    # ────────────────────────────────────────────
    # 序列化
    # ────────────────────────────────────────────

    def to_dict(self) -> dict:
        """转换为可直接 json.dumps 的字典。"""
        return {
            "session_id": self.session_id,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "turn_count": self.turn_count,
            "messages": self.messages,
            "total_usage": self.total_usage,
            "system_prompt": self.system_prompt,
            "metadata": self.metadata,
        }

    @classmethod
    def from_dict(cls, data: dict) -> "SessionSnapshot":
        """从字典反序列化。反序列化时验证必要字段。"""
        required = {"session_id", "created_at", "updated_at", "turn_count", "messages", "total_usage"}
        missing = required - data.keys()
        if missing:
            raise ValueError(f"快照数据缺少字段: {missing}")

        # 确保 total_usage 结构正确
        usage = data["total_usage"]
        for key in ("prompt_tokens", "completion_tokens", "total_tokens"):
            usage.setdefault(key, 0)

        # 确保 messages 中 tool_calls 字段类型正确
        messages = []
        for msg in data["messages"]:
            msg = dict(msg)  # 拷贝，避免修改原始数据
            if "tool_calls" in msg and msg["tool_calls"] is not None:
                # tool_calls 必须是 dict 列表
                if not isinstance(msg["tool_calls"], list):
                    msg["tool_calls"] = []
            messages.append(msg)

        return cls(
            session_id=data["session_id"],
            created_at=data["created_at"],
            updated_at=data["updated_at"],
            turn_count=data["turn_count"],
            messages=messages,
            total_usage=usage,
            system_prompt=data.get("system_prompt", ""),
            metadata=data.get("metadata", {}),
        )

    # ────────────────────────────────────────────
    # 工具方法
    # ────────────────────────────────────────────

    @property
    def message_count(self) -> int:
        """不含 system 消息的消息数量。"""
        return sum(1 for m in self.messages if m.get("role") != "system")

    def summary(self) -> dict:
        """用于列表展示的摘要信息。"""
        return {
            "session_id": self.session_id,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "turn_count": self.turn_count,
            "message_count": self.message_count,
            "total_tokens": self.total_usage.get("total_tokens", 0),
        }

    def __repr__(self) -> str:
        return (
            f"SessionSnapshot(id={self.session_id!r}, "
            f"turns={self.turn_count}, messages={self.message_count})"
        )


print("SessionSnapshot 定义完成")

### 测试 SessionSnapshot 序列化

In [ ]:
# ── 测试：序列化往返（to_dict → from_dict）
now = datetime.now(timezone.utc).isoformat()

sample_messages = [
    {"role": "system", "content": "你是一个助手"},
    {"role": "user", "content": "帮我写一个函数"},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "call_001",
                "type": "function",
                "function": {"name": "write_file", "arguments": '{"path": "foo.py", "content": "def foo(): pass"}'},
            }
        ],
    },
    {"role": "tool", "content": "写入成功", "tool_call_id": "call_001"},
    {"role": "assistant", "content": "函数已写入 foo.py"},
]

original = SessionSnapshot(
    session_id="test-001",
    created_at=now,
    updated_at=now,
    turn_count=2,
    messages=sample_messages,
    total_usage={"prompt_tokens": 150, "completion_tokens": 80, "total_tokens": 230},
    system_prompt="你是一个助手",
)

# 序列化 → JSON 字符串 → 反序列化
raw_dict = original.to_dict()
json_str = json.dumps(raw_dict, ensure_ascii=False, indent=2)
restored = SessionSnapshot.from_dict(json.loads(json_str))

assert restored.session_id == original.session_id
assert restored.turn_count == original.turn_count
assert len(restored.messages) == len(original.messages)
assert restored.total_usage["total_tokens"] == 230
assert restored.message_count == 4  # 不含 system
assert isinstance(restored.messages[2]["tool_calls"], list)
assert restored.messages[2]["tool_calls"][0]["id"] == "call_001"

print("序列化往返测试通过")
print(f"摘要: {restored.summary()}")

In [ ]:
# ── 测试：from_dict 错误处理
try:
    bad = SessionSnapshot.from_dict({"session_id": "x", "turn_count": 1})
except ValueError as e:
    print(f"缺少字段时正确抛出错误: {e}")

# 测试 tool_calls 为非列表时的修复
bad_msg = {
    "session_id": "test-002",
    "created_at": now,
    "updated_at": now,
    "turn_count": 1,
    "messages": [{"role": "assistant", "content": "hi", "tool_calls": "wrong_type"}],
    "total_usage": {},
}
fixed = SessionSnapshot.from_dict(bad_msg)
assert fixed.messages[0]["tool_calls"] == []  # 修复为空列表
print("tool_calls 类型修复测试通过")

## 2. PersistenceManager

`PersistenceManager` 负责读写磁盘。它不依赖活跃的 `Session` 对象，只操作 `SessionSnapshot`——这样即使 `Session` 类发生变化，持久化层也保持稳定。

```
PersistenceManager
    ├── save_session(session)      → .ai_agent/sessions/{id}.json  (覆盖)
    ├── load_session(id)           → SessionSnapshot | None
    ├── list_sessions()            → [摘要列表]
    ├── save_checkpoint(session)   → .ai_agent/checkpoints/{id}_{ts}.json (追加)
    ├── load_checkpoint(ck_id)     → SessionSnapshot | None
    ├── list_checkpoints(id?)      → [摘要列表]
    └── restore_session(session, snapshot)
```

文件权限设为 600（仅所有者读写），防止其他用户读取对话内容。

In [ ]:
import asyncio
import stat
import uuid
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional
import json
import os


class PersistenceManager:
    """负责把 Session 状态持久化到磁盘，以及从磁盘还原。"""

    def __init__(self, base_dir: Path = Path(".ai_agent")):
        self.base_dir = Path(base_dir)
        self.sessions_dir = self.base_dir / "sessions"
        self.checkpoints_dir = self.base_dir / "checkpoints"
        self._ensure_dirs()

    def _ensure_dirs(self):
        """创建必要的目录结构（幂等）。"""
        self.sessions_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoints_dir.mkdir(parents=True, exist_ok=True)

    def _now_iso(self) -> str:
        """返回带时区的 ISO 8601 时间字符串。"""
        return datetime.now(timezone.utc).isoformat()

    def _write_json_secure(self, path: Path, data: dict):
        """写入 JSON 文件，权限设为 600。"""
        path.parent.mkdir(parents=True, exist_ok=True)
        json_str = json.dumps(data, ensure_ascii=False, indent=2)
        path.write_text(json_str, encoding="utf-8")
        # 设置权限 600：仅所有者可读写
        path.chmod(stat.S_IRUSR | stat.S_IWUSR)

    def _read_json(self, path: Path) -> Optional[dict]:
        """读取 JSON 文件，返回字典。文件不存在或解析失败时返回 None。"""
        if not path.exists():
            return None
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as e:
            print(f"读取文件失败 {path}: {e}")
            return None

    def _session_to_snapshot(self, session) -> SessionSnapshot:
        """
        从 Session 对象提取快照。
        Session 需要有：
          - session_id (str)
          - created_at (str)  可选，没有则使用 now
          - turn_count (int)
          - context_manager (ContextManager)
          - total_usage (dict 或 UsageStats)
        """
        now = self._now_iso()

        # 兼容 session_id 不存在的情况（自动生成）
        session_id = getattr(session, "session_id", None) or str(uuid.uuid4())
        created_at = getattr(session, "created_at", now)
        turn_count = getattr(session, "turn_count", 0)

        # 提取消息
        ctx = getattr(session, "context_manager", None)
        if ctx is not None:
            messages = ctx.get_messages()
            system_prompt = getattr(ctx, "system_prompt", "")
        else:
            messages = []
            system_prompt = ""

        # 提取 usage（兼容 dict 和 UsageStats 对象）
        raw_usage = getattr(session, "total_usage", {})
        if hasattr(raw_usage, "__dict__"):
            total_usage = {
                "prompt_tokens": getattr(raw_usage, "prompt_tokens", 0),
                "completion_tokens": getattr(raw_usage, "completion_tokens", 0),
                "total_tokens": getattr(raw_usage, "total_tokens", 0),
            }
        elif isinstance(raw_usage, dict):
            total_usage = {
                "prompt_tokens": raw_usage.get("prompt_tokens", 0),
                "completion_tokens": raw_usage.get("completion_tokens", 0),
                "total_tokens": raw_usage.get("total_tokens", 0),
            }
        else:
            total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

        return SessionSnapshot(
            session_id=session_id,
            created_at=created_at,
            updated_at=now,
            turn_count=turn_count,
            messages=messages,
            total_usage=total_usage,
            system_prompt=system_prompt,
        )

    # ────────────────────────────────────────────
    # 会话（sessions）
    # ────────────────────────────────────────────

    async def save_session(self, session) -> Path:
        """把 session 当前状态保存到 sessions/{session_id}.json。覆盖更新。"""
        snapshot = self._session_to_snapshot(session)
        path = self.sessions_dir / f"{snapshot.session_id}.json"
        self._write_json_secure(path, snapshot.to_dict())
        return path

    async def load_session(self, session_id: str) -> Optional[SessionSnapshot]:
        """加载会话快照，不存在时返回 None。"""
        path = self.sessions_dir / f"{session_id}.json"
        data = self._read_json(path)
        if data is None:
            return None
        try:
            return SessionSnapshot.from_dict(data)
        except ValueError as e:
            print(f"加载会话 {session_id} 失败: {e}")
            return None

    def list_sessions(self) -> list:
        """返回所有已保存会话的摘要，按 updated_at 倒序排列。"""
        result = []
        for path in self.sessions_dir.glob("*.json"):
            data = self._read_json(path)
            if data is None:
                continue
            try:
                snap = SessionSnapshot.from_dict(data)
                result.append(snap.summary())
            except ValueError:
                continue
        result.sort(key=lambda x: x["updated_at"], reverse=True)
        return result

    # ────────────────────────────────────────────
    # 检查点（checkpoints）
    # ────────────────────────────────────────────

    async def save_checkpoint(self, session) -> str:
        """保存检查点，返回 checkpoint_id（文件名不含路径和 .json 后缀）。"""
        snapshot = self._session_to_snapshot(session)
        # 时间戳使用本地时间（显示友好），精确到秒
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        checkpoint_id = f"{snapshot.session_id}_{ts}"
        path = self.checkpoints_dir / f"{checkpoint_id}.json"

        # 防止同一秒内重复（追加毫秒）
        if path.exists():
            ts_ms = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:20]
            checkpoint_id = f"{snapshot.session_id}_{ts_ms}"
            path = self.checkpoints_dir / f"{checkpoint_id}.json"

        # 在 metadata 中记录 checkpoint_id
        snapshot.metadata["checkpoint_id"] = checkpoint_id
        self._write_json_secure(path, snapshot.to_dict())
        return checkpoint_id

    async def load_checkpoint(self, checkpoint_id: str) -> Optional[SessionSnapshot]:
        """加载检查点快照。checkpoint_id 可以不带 .json 后缀。"""
        # 兼容带/不带 .json 后缀
        if checkpoint_id.endswith(".json"):
            checkpoint_id = checkpoint_id[:-5]
        path = self.checkpoints_dir / f"{checkpoint_id}.json"
        data = self._read_json(path)
        if data is None:
            return None
        try:
            return SessionSnapshot.from_dict(data)
        except ValueError as e:
            print(f"加载检查点 {checkpoint_id} 失败: {e}")
            return None

    def list_checkpoints(self, session_id: Optional[str] = None) -> list:
        """列出检查点。session_id=None 返回全部，否则只返回该会话的。"""
        result = []
        for path in self.checkpoints_dir.glob("*.json"):
            data = self._read_json(path)
            if data is None:
                continue
            try:
                snap = SessionSnapshot.from_dict(data)
                if session_id is not None and snap.session_id != session_id:
                    continue
                summary = snap.summary()
                summary["checkpoint_id"] = snap.metadata.get(
                    "checkpoint_id", path.stem  # 回退到文件名
                )
                result.append(summary)
            except ValueError:
                continue
        result.sort(key=lambda x: x["updated_at"], reverse=True)
        return result

    # ────────────────────────────────────────────
    # 恢复
    # ────────────────────────────────────────────

    async def restore_session(self, session, snapshot: SessionSnapshot):
        """
        把 snapshot 的状态恢复到 session 中。
        修改：context_manager 的消息历史、turn_count、total_usage。
        """
        ctx = getattr(session, "context_manager", None)
        if ctx is None:
            raise AttributeError("session 没有 context_manager 属性")

        # 清空当前消息历史
        ctx.clear_messages()

        # 逐条重放消息（跳过 system 消息，因为 clear_messages 后 system 已在 ctx 中）
        for msg in snapshot.messages:
            role = msg.get("role")
            content = msg.get("content", "")

            if role == "system":
                # system prompt 由 ContextManager 自己管理，跳过
                continue
            elif role == "user":
                ctx.add_user_message(content)
            elif role == "assistant":
                tool_calls = msg.get("tool_calls")
                ctx.add_assistant_message(content, tool_calls=tool_calls)
            elif role == "tool":
                tool_call_id = msg.get("tool_call_id", "")
                ctx.add_tool_result(tool_call_id, content)
            # 未知 role 直接跳过

        # 恢复 turn_count
        session.turn_count = snapshot.turn_count

        # 恢复 total_usage
        raw = snapshot.total_usage
        existing_usage = getattr(session, "total_usage", None)
        if hasattr(existing_usage, "prompt_tokens"):
            # UsageStats 对象：直接赋值属性
            existing_usage.prompt_tokens = raw.get("prompt_tokens", 0)
            existing_usage.completion_tokens = raw.get("completion_tokens", 0)
            existing_usage.total_tokens = raw.get("total_tokens", 0)
        else:
            # dict 或 None：直接替换
            session.total_usage = dict(raw)


print("PersistenceManager 定义完成")

## 3. 轻量 Session 对象（用于测试）

在没有完整 `LLMClient` 的情况下，我们用一个轻量 `Session` 桩来验证持久化逻辑。它复现了前几章 `Session` 的关键接口。

In [ ]:
import sys
sys.path.insert(0, "..")

# 尝试导入真实的 ContextManager；失败时用桩替代
try:
    from src.context_manager import ContextManager, build_system_prompt
    print("使用真实 ContextManager")
except ImportError:
    # ── 桩：复现 ContextManager 的最小接口 ──
    class ContextManager:
        def __init__(self, system_prompt: str = ""):
            self.system_prompt = system_prompt
            self._messages = []
            if system_prompt:
                self._messages.append({"role": "system", "content": system_prompt})

        def add_user_message(self, content: str):
            self._messages.append({"role": "user", "content": content})

        def add_assistant_message(self, content, tool_calls=None):
            msg = {"role": "assistant", "content": content}
            if tool_calls:
                msg["tool_calls"] = tool_calls
            self._messages.append(msg)

        def add_tool_result(self, tool_call_id: str, content: str):
            self._messages.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call_id,
            })

        def get_messages(self) -> list:
            return list(self._messages)

        def clear_messages(self):
            self._messages = []
            if self.system_prompt:
                self._messages.append({"role": "system", "content": self.system_prompt})

        def update_usage(self, prompt_tokens: int, completion_tokens: int):
            pass

    def build_system_prompt(working_directory=".") -> str:
        return "你是一个 AI 助手"

    print("使用桩版 ContextManager（src/ 不在路径中）")


@dataclass
class _UsageStats:
    """轻量版 UsageStats，仅用于本章测试。"""
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

    def add(self, prompt: int, completion: int):
        self.prompt_tokens += prompt
        self.completion_tokens += completion
        self.total_tokens += prompt + completion


class Session:
    """轻量 Session 桩，复现前几章 Session 的关键接口。"""

    def __init__(self, session_id: Optional[str] = None, system_prompt: str = ""):
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.created_at = datetime.now(timezone.utc).isoformat()
        self.turn_count = 0
        self.total_usage = _UsageStats()
        self.context_manager = ContextManager(system_prompt or build_system_prompt())

    def add_turn(self, user_msg: str, assistant_msg: str,
                 prompt_tokens=10, completion_tokens=10):
        """模拟一轮对话（用于测试）。"""
        self.context_manager.add_user_message(user_msg)
        self.context_manager.add_assistant_message(assistant_msg)
        self.turn_count += 1
        self.total_usage.add(prompt_tokens, completion_tokens)

    def message_count(self) -> int:
        msgs = self.context_manager.get_messages()
        return sum(1 for m in msgs if m["role"] != "system")

    def __repr__(self) -> str:
        return f"Session(id={self.session_id!r}, turns={self.turn_count}, msgs={self.message_count()})"


print("Session 桩定义完成")

## 4. 测试：保存与加载会话

In [ ]:
import tempfile

# 使用临时目录避免污染工作区
TEST_DIR = Path(tempfile.mkdtemp()) / ".ai_agent"
pm = PersistenceManager(base_dir=TEST_DIR)

print(f"测试目录: {TEST_DIR}")
print(f"sessions 目录: {pm.sessions_dir}")
print(f"checkpoints 目录: {pm.checkpoints_dir}")

In [ ]:
# ── 测试：保存会话 ──

session = Session(session_id="demo-001", system_prompt="你是一个代码助手")
session.add_turn("写一个 hello world", "print('Hello, World!')", 50, 20)
session.add_turn("加个注释", "# 打印问候语\nprint('Hello, World!')", 80, 30)

print("保存前:", session)

async def test_save():
    path = await pm.save_session(session)
    print(f"保存到: {path}")
    print(f"文件存在: {path.exists()}")

    # 检查文件权限
    file_stat = path.stat()
    mode = oct(file_stat.st_mode & 0o777)
    print(f"文件权限: {mode}")
    assert mode == "0o600", f"权限应为 0o600，实际为 {mode}"

    # 检查内容可读
    data = json.loads(path.read_text())
    print(f"JSON 字段: {list(data.keys())}")
    assert data["session_id"] == "demo-001"
    assert data["turn_count"] == 2
    print("保存会话测试通过")

await test_save()

In [ ]:
# ── 测试：加载会话 ──

async def test_load():
    snap = await pm.load_session("demo-001")
    assert snap is not None
    print(f"加载成功: {snap}")
    print(f"消息数量: {snap.message_count}")
    print(f"token 消耗: {snap.total_usage}")
    assert snap.session_id == "demo-001"
    assert snap.turn_count == 2
    # 2 轮对话 = 4 条消息（user + assistant × 2），system 不计入 message_count
    assert snap.message_count == 4
    print("加载会话测试通过")

    # 加载不存在的会话
    missing = await pm.load_session("not-exist")
    assert missing is None
    print("加载不存在会话返回 None：正确")

await test_load()

In [ ]:
# ── 测试：list_sessions ──

# 再保存几个会话
async def test_list_sessions():
    for i in range(2, 5):
        s = Session(session_id=f"demo-00{i}")
        s.add_turn(f"问题 {i}", f"回答 {i}")
        await pm.save_session(s)

    sessions = pm.list_sessions()
    print(f"共有 {len(sessions)} 个会话:")
    for s in sessions:
        print(f"  {s['session_id']} | turns={s['turn_count']} | msgs={s['message_count']} | updated={s['updated_at'][:19]}")

    assert len(sessions) >= 4
    # 验证按 updated_at 倒序
    dates = [s["updated_at"] for s in sessions]
    assert dates == sorted(dates, reverse=True), "应按 updated_at 倒序排列"
    print("list_sessions 测试通过")

await test_list_sessions()

## 5. 测试：检查点保存与恢复

检查点的核心用途是**状态回退**——不是「继续上次」，而是「撤销到之前某个时刻」。

```
状态 A ──/checkpoint→ ck_A
    │
    ▼
状态 B ──/checkpoint→ ck_B
    │
    ▼
状态 C（出错或不满意）
    │
    ▼
/restore ck_A → 状态恢复为 A
```

In [ ]:
# ── 测试：保存检查点 ──

async def test_checkpoint():
    session = Session(session_id="ck-demo", system_prompt="你是一个助手")

    # 第一轮对话后保存检查点 A
    session.add_turn("帮我写一个排序函数", "def sort_list(lst): return sorted(lst)", 60, 30)
    ck_a = await pm.save_checkpoint(session)
    print(f"检查点 A: {ck_a}")
    state_a_turns = session.turn_count
    state_a_msg_count = session.message_count()

    # 继续对话
    session.add_turn("用快排实现", "def quicksort(arr): ...", 80, 50)
    session.add_turn("加测试用例", "assert quicksort([3,1,2]) == [1,2,3]", 90, 40)
    ck_b = await pm.save_checkpoint(session)
    print(f"检查点 B: {ck_b}")

    print(f"当前状态: turns={session.turn_count}, msgs={session.message_count()}")
    print(f"检查点 A 状态: turns={state_a_turns}, msgs={state_a_msg_count}")

    # 加载检查点 A
    snap_a = await pm.load_checkpoint(ck_a)
    assert snap_a is not None
    assert snap_a.turn_count == state_a_turns
    assert snap_a.message_count == state_a_msg_count
    print(f"加载检查点 A 成功: turns={snap_a.turn_count}, msgs={snap_a.message_count}")

    return ck_a, ck_b, state_a_turns, state_a_msg_count

ck_a, ck_b, expected_turns, expected_msgs = await test_checkpoint()

In [ ]:
# ── 测试：list_checkpoints ──

async def test_list_checkpoints():
    checkpoints = pm.list_checkpoints(session_id="ck-demo")
    print(f"会话 ck-demo 的检查点数量: {len(checkpoints)}")
    for ck in checkpoints:
        print(f"  {ck['checkpoint_id']} | turns={ck['turn_count']} | msgs={ck['message_count']}")
    assert len(checkpoints) >= 2

    # 测试 session_id=None 返回所有检查点
    all_ck = pm.list_checkpoints()
    print(f"全部检查点数量: {len(all_ck)}")
    assert len(all_ck) >= len(checkpoints)
    print("list_checkpoints 测试通过")

await test_list_checkpoints()

In [ ]:
# ── 测试：restore_session（核心功能）──

async def test_restore():
    # 创建一个新 session，模拟「重新启动后」的空白状态
    session = Session(session_id="ck-demo", system_prompt="你是一个助手")
    print(f"恢复前（空白）: turns={session.turn_count}, msgs={session.message_count()}")

    # 加载检查点 A
    snap_a = await pm.load_checkpoint(ck_a)
    assert snap_a is not None

    # 恢复到检查点 A
    await pm.restore_session(session, snap_a)
    print(f"恢复后（检查点 A）: turns={session.turn_count}, msgs={session.message_count()}")

    assert session.turn_count == expected_turns, (
        f"turn_count 应为 {expected_turns}，实际为 {session.turn_count}"
    )
    assert session.message_count() == expected_msgs, (
        f"message_count 应为 {expected_msgs}，实际为 {session.message_count()}"
    )

    # 验证消息内容
    msgs = session.context_manager.get_messages()
    non_system = [m for m in msgs if m["role"] != "system"]
    print("恢复后的消息内容:")
    for m in non_system:
        content_preview = str(m.get("content", ""))[:50]
        print(f"  [{m['role']}] {content_preview}")

    print("restore_session 测试通过")

await test_restore()

## 6. 完整演示：检查点回退工作流

这个演示模拟实际使用场景：

1. 开始对话
2. 在某个节点打检查点
3. 继续对话（修改状态）
4. 恢复到检查点
5. 验证状态已回退

In [ ]:
async def full_demo():
    print("=" * 55)
    print("完整演示：检查点回退工作流")
    print("=" * 55)

    pm_demo = PersistenceManager(base_dir=TEST_DIR / "demo")
    session = Session(session_id="workflow-demo", system_prompt="你是一个代码助手")

    # ── 阶段 1：初始对话 ──────────────────────────
    print("\n[阶段 1] 初始对话")
    session.add_turn(
        "用 Python 写一个冒泡排序",
        "def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(n-i-1):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n    return arr",
        prompt_tokens=80, completion_tokens=60,
    )
    print(f"  当前状态: turns={session.turn_count}, msgs={session.message_count()}")
    print(f"  tokens: {session.total_usage.total_tokens}")

    # ── 阶段 2：保存检查点 ────────────────────────
    print("\n[阶段 2] /checkpoint")
    ck_id = await pm_demo.save_checkpoint(session)
    print(f"  检查点 ID: {ck_id}")
    snapshot_before = (
        session.turn_count,
        session.message_count(),
        session.total_usage.total_tokens,
    )

    # ── 阶段 3：继续对话（修改状态）────────────────
    print("\n[阶段 3] 继续对话（修改状态）")
    session.add_turn("加一个单元测试", "import unittest\nclass TestBubbleSort(unittest.TestCase): ...",
                     prompt_tokens=100, completion_tokens=80)
    session.add_turn("再加文档字符串", "def bubble_sort(arr):\n    \"\"\"对列表原地排序...\"\"\"\n    ...",
                     prompt_tokens=120, completion_tokens=90)
    print(f"  修改后: turns={session.turn_count}, msgs={session.message_count()}")
    print(f"  tokens: {session.total_usage.total_tokens}")

    # ── 阶段 4：/restore 回到检查点 ──────────────
    print("\n[阶段 4] /restore", ck_id)
    snap = await pm_demo.load_checkpoint(ck_id)
    assert snap is not None, "检查点加载失败"
    await pm_demo.restore_session(session, snap)
    print(f"  恢复后: turns={session.turn_count}, msgs={session.message_count()}")
    print(f"  tokens: {session.total_usage.total_tokens}")

    # ── 阶段 5：验证 ──────────────────────────────
    print("\n[阶段 5] 验证")
    restored = (
        session.turn_count,
        session.message_count(),
        session.total_usage.total_tokens,
    )
    assert restored == snapshot_before, (
        f"状态不匹配\n  期望: {snapshot_before}\n  实际: {restored}"
    )
    print("  turn_count 匹配")
    print("  message_count 匹配")
    print("  total_tokens 匹配")
    print("\n  [通过] 检查点回退工作流验证完成")

    # 查看恢复后的消息内容
    msgs = [m for m in session.context_manager.get_messages() if m["role"] != "system"]
    print(f"\n  恢复后的消息 ({len(msgs)} 条):")
    for m in msgs:
        print(f"    [{m['role']}] {str(m.get('content',''))[:60]}")

await full_demo()

## 7. CLI 命令处理器

把持久化功能接入 REPL 循环。这里实现独立的命令处理函数，可以直接嵌入前几章的 `run_session_loop`。

```
命令        作用
/save       保存当前会话
/sessions   列出所有已保存的会话
/resume <id>  加载并恢复会话
/checkpoint   创建检查点
/restore <id> 恢复到检查点
```

In [ ]:
from typing import Callable, Awaitable


class PersistenceCLI:
    """把 PersistenceManager 功能封装为 /command 风格的处理器。"""

    def __init__(self, manager: PersistenceManager):
        self.manager = manager

    async def handle(self, command: str, session: Session) -> bool:
        """
        处理持久化相关命令。
        返回 True 表示命令已处理，False 表示不是持久化命令。
        """
        parts = command.strip().split()
        if not parts or not parts[0].startswith("/"):
            return False

        cmd = parts[0].lower()
        args = parts[1:]

        if cmd == "/save":
            await self._cmd_save(session)
        elif cmd == "/sessions":
            await self._cmd_sessions()
        elif cmd == "/resume":
            await self._cmd_resume(args, session)
        elif cmd == "/checkpoint":
            await self._cmd_checkpoint(session)
        elif cmd == "/restore":
            await self._cmd_restore(args, session)
        else:
            return False  # 不是持久化命令

        return True

    async def _cmd_save(self, session: Session):
        try:
            path = await self.manager.save_session(session)
            print(f"已保存会话 [{session.session_id}] → {path}")
        except Exception as e:
            print(f"保存失败: {e}")

    async def _cmd_sessions(self):
        sessions = self.manager.list_sessions()
        if not sessions:
            print("没有已保存的会话")
            return
        print(f"{'ID':<20} {'对话轮数':>8} {'消息数':>6} {'最后更新':>20}")
        print("-" * 58)
        for s in sessions:
            updated = s["updated_at"][:19].replace("T", " ")
            print(f"{s['session_id']:<20} {s['turn_count']:>8} {s['message_count']:>6} {updated:>20}")

    async def _cmd_resume(self, args: list, session: Session):
        if not args:
            print("用法: /resume <session_id>")
            return
        session_id = args[0]
        snap = await self.manager.load_session(session_id)
        if snap is None:
            print(f"找不到会话: {session_id}")
            return
        try:
            await self.manager.restore_session(session, snap)
            print(f"已恢复会话 [{session_id}]，共 {snap.turn_count} 轮对话")
        except Exception as e:
            print(f"恢复失败: {e}")

    async def _cmd_checkpoint(self, session: Session):
        try:
            ck_id = await self.manager.save_checkpoint(session)
            print(f"检查点已保存: {ck_id}")
        except Exception as e:
            print(f"保存检查点失败: {e}")

    async def _cmd_restore(self, args: list, session: Session):
        if not args:
            # 没有指定 ID，列出可用的检查点
            checkpoints = self.manager.list_checkpoints(session_id=session.session_id)
            if not checkpoints:
                print(f"会话 [{session.session_id}] 没有检查点")
            else:
                print("可用检查点:")
                for ck in checkpoints:
                    print(f"  {ck['checkpoint_id']} | turns={ck['turn_count']}")
            return

        checkpoint_id = args[0]
        snap = await self.manager.load_checkpoint(checkpoint_id)
        if snap is None:
            print(f"找不到检查点: {checkpoint_id}")
            return
        try:
            await self.manager.restore_session(session, snap)
            print(f"已恢复到检查点 [{checkpoint_id}]，共 {snap.turn_count} 轮对话")
        except Exception as e:
            print(f"恢复失败: {e}")


print("PersistenceCLI 定义完成")

### 测试 CLI 命令处理器

In [ ]:
async def test_cli():
    pm_cli = PersistenceManager(base_dir=TEST_DIR / "cli")
    cli = PersistenceCLI(pm_cli)

    session = Session(session_id="cli-test", system_prompt="测试会话")
    session.add_turn("第一轮", "回答一", 40, 20)

    print("── /save ──")
    handled = await cli.handle("/save", session)
    assert handled is True

    print("\n── /checkpoint ──")
    await cli.handle("/checkpoint", session)

    # 继续对话
    session.add_turn("第二轮", "回答二", 50, 25)
    session.add_turn("第三轮", "回答三", 60, 30)

    print("\n── /checkpoint（第二个）──")
    await cli.handle("/checkpoint", session)

    print("\n── /sessions ──")
    await cli.handle("/sessions", session)

    print("\n── /restore（无参数，列出可用检查点）──")
    await cli.handle("/restore", session)

    print("\n── /restore <不存在的 ID> ──")
    await cli.handle("/restore not-exist-ck", session)

    print("\n── 非持久化命令 ──")
    handled = await cli.handle("普通消息", session)
    assert handled is False
    print("非命令返回 False：正确")

await test_cli()

## 8. 错误处理场景

持久化层必须优雅处理各种故障——文件损坏、权限问题、并发写入等。

In [ ]:
async def test_error_cases():
    pm_err = PersistenceManager(base_dir=TEST_DIR / "errors")

    # ── 错误 1：加载不存在的会话 ──
    result = await pm_err.load_session("ghost-session")
    assert result is None
    print("[通过] 加载不存在会话返回 None")

    # ── 错误 2：加载不存在的检查点 ──
    result = await pm_err.load_checkpoint("ghost-checkpoint")
    assert result is None
    print("[通过] 加载不存在检查点返回 None")

    # ── 错误 3：文件内容损坏 ──
    corrupt_path = pm_err.sessions_dir / "corrupt.json"
    corrupt_path.parent.mkdir(parents=True, exist_ok=True)
    corrupt_path.write_text("这不是有效的 JSON { 内容", encoding="utf-8")
    result = await pm_err.load_session("corrupt")
    assert result is None
    print("[通过] 损坏的 JSON 文件返回 None")

    # ── 错误 4：快照字段不完整 ──
    incomplete_path = pm_err.sessions_dir / "incomplete.json"
    incomplete_data = {"session_id": "incomplete", "turn_count": 1}  # 缺少必要字段
    incomplete_path.write_text(json.dumps(incomplete_data), encoding="utf-8")
    result = await pm_err.load_session("incomplete")
    assert result is None
    print("[通过] 不完整的快照数据返回 None")

    # ── 错误 5：restore_session 目标 session 没有 context_manager ──
    class BrokenSession:
        session_id = "broken"
        turn_count = 0
        total_usage = {}
        # 故意没有 context_manager

    session = Session(session_id="good", system_prompt="测试")
    session.add_turn("问", "答")
    ck_id = await pm_err.save_checkpoint(session)
    snap = await pm_err.load_checkpoint(ck_id)

    try:
        await pm_err.restore_session(BrokenSession(), snap)
        print("[失败] 应当抛出 AttributeError")
    except AttributeError as e:
        print(f"[通过] 没有 context_manager 时正确抛出: {e}")

    # ── 错误 6：list_sessions 遇到损坏文件时跳过 ──
    sessions = pm_err.list_sessions()
    valid_ids = [s["session_id"] for s in sessions]
    assert "corrupt" not in valid_ids
    assert "incomplete" not in valid_ids
    print(f"[通过] list_sessions 跳过损坏文件，返回 {len(sessions)} 个有效会话")

await test_error_cases()

## 9. 与 Ollama 的集成演示

如果本地运行着 Ollama，这一节展示与真实 LLM 的完整集成。会话保存后可以在另一个进程（或下次启动）中继续。

> 如果没有 Ollama，这个 cell 会捕获连接错误并跳过，不影响其他测试。

In [ ]:
import httpx

async def ollama_integration_demo():
    """与 Ollama 集成的完整演示（需要本地运行 Ollama）。"""

    # 检查 Ollama 是否可用
    try:
        async with httpx.AsyncClient(timeout=3.0) as client:
            resp = await client.get("http://localhost:11434/api/tags")
            if resp.status_code != 200:
                print("Ollama 未运行，跳过集成演示")
                return
        print("Ollama 可用，开始集成演示")
    except Exception:
        print("Ollama 未运行，跳过集成演示")
        return

    # 尝试导入真实组件
    try:
        from src.llm_client import LLMClient
        from src.context_manager import ContextManager, build_system_prompt
    except ImportError:
        print("src/ 组件未找到，跳过 Ollama 演示")
        return

    # ── 构建真实 Session 类 ──
    class RealSession:
        def __init__(self, session_id=None):
            self.session_id = session_id or str(uuid.uuid4())[:8]
            self.created_at = datetime.now(timezone.utc).isoformat()
            self.turn_count = 0
            self.context_manager = ContextManager(build_system_prompt())
            self.llm = LLMClient()

            class _Usage:
                prompt_tokens = 0
                completion_tokens = 0
                total_tokens = 0
                def add(self, p, c):
                    self.prompt_tokens += p
                    self.completion_tokens += c
                    self.total_tokens += p + c

            self.total_usage = _Usage()

        async def chat(self, user_input: str) -> str:
            self.context_manager.add_user_message(user_input)
            response, usage = await self.llm.chat_completion(
                self.context_manager.get_messages()
            )
            self.context_manager.add_assistant_message(response)
            self.total_usage.add(usage.prompt_tokens, usage.completion_tokens)
            self.turn_count += 1
            return response

        async def close(self):
            await self.llm.close()

    pm_real = PersistenceManager(base_dir=TEST_DIR / "ollama")
    session = RealSession(session_id="ollama-demo")

    try:
        print("\n[轮次 1] 发送消息...")
        resp1 = await session.chat("用一句话解释什么是递归")
        print(f"  助手: {resp1[:80]}..." if len(resp1) > 80 else f"  助手: {resp1}")
        print(f"  tokens: {session.total_usage.total_tokens}")

        # 保存检查点
        ck_id = await pm_real.save_checkpoint(session)
        print(f"\n[检查点] {ck_id}")

        print("\n[轮次 2] 继续对话...")
        resp2 = await session.chat("给一个 Python 例子")
        print(f"  助手: {resp2[:80]}..." if len(resp2) > 80 else f"  助手: {resp2}")
        print(f"  turn_count: {session.turn_count}, tokens: {session.total_usage.total_tokens}")

        # 恢复到检查点
        print(f"\n[恢复] 恢复到检查点 {ck_id}...")
        snap = await pm_real.load_checkpoint(ck_id)
        await pm_real.restore_session(session, snap)
        print(f"  恢复后: turn_count={session.turn_count}, msgs={len([m for m in session.context_manager.get_messages() if m['role']!='system'])}")

        # 从检查点状态继续（走不同的方向）
        print("\n[轮次 2b] 从检查点继续（不同方向）...")
        resp2b = await session.chat("用 JavaScript 举个例子")
        print(f"  助手: {resp2b[:80]}..." if len(resp2b) > 80 else f"  助手: {resp2b}")

        # 保存最终会话
        save_path = await pm_real.save_session(session)
        print(f"\n[保存] 会话已保存到 {save_path}")

        print("\nOllama 集成演示完成")
    finally:
        await session.close()

await ollama_integration_demo()

## 10. 保存为 src/persistence.py

In [ ]:
import os

# 创建 src/ 目录（相对于 notebook 所在的 course/ 目录向上一层）
src_dir = Path("..") / "src"
src_dir.mkdir(exist_ok=True)

persistence_code = '''
"""
persistence.py — 会话持久化与检查点管理

两种持久化粒度：
  session:    保存整个对话历史，可以跨时间恢复继续
  checkpoint: 某个时间点的快照，可以回退到之前的状态

文件布局：
  .ai_agent/
  ├── sessions/{session_id}.json
  └── checkpoints/{session_id}_{YYYYMMDD_HHMMSS}.json
"""

import json
import os
import stat
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional


@dataclass
class SessionSnapshot:
    """会话状态的完整快照，可序列化为 JSON。"""

    session_id: str
    created_at: str          # ISO 8601 格式，带时区
    updated_at: str
    turn_count: int
    messages: list           # ctx.get_messages() 的原始结果
    total_usage: dict        # {"prompt_tokens": int, "completion_tokens": int, "total_tokens": int}
    system_prompt: str = ""
    metadata: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        """转换为可直接 json.dumps 的字典。"""
        return {
            "session_id": self.session_id,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "turn_count": self.turn_count,
            "messages": self.messages,
            "total_usage": self.total_usage,
            "system_prompt": self.system_prompt,
            "metadata": self.metadata,
        }

    @classmethod
    def from_dict(cls, data: dict) -> "SessionSnapshot":
        """从字典反序列化。反序列化时验证必要字段。"""
        required = {"session_id", "created_at", "updated_at", "turn_count", "messages", "total_usage"}
        missing = required - data.keys()
        if missing:
            raise ValueError(f"快照数据缺少字段: {missing}")

        usage = data["total_usage"]
        for key in ("prompt_tokens", "completion_tokens", "total_tokens"):
            usage.setdefault(key, 0)

        messages = []
        for msg in data["messages"]:
            msg = dict(msg)
            if "tool_calls" in msg and msg["tool_calls"] is not None:
                if not isinstance(msg["tool_calls"], list):
                    msg["tool_calls"] = []
            messages.append(msg)

        return cls(
            session_id=data["session_id"],
            created_at=data["created_at"],
            updated_at=data["updated_at"],
            turn_count=data["turn_count"],
            messages=messages,
            total_usage=usage,
            system_prompt=data.get("system_prompt", ""),
            metadata=data.get("metadata", {}),
        )

    @property
    def message_count(self) -> int:
        """不含 system 消息的消息数量。"""
        return sum(1 for m in self.messages if m.get("role") != "system")

    def summary(self) -> dict:
        """用于列表展示的摘要信息。"""
        return {
            "session_id": self.session_id,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "turn_count": self.turn_count,
            "message_count": self.message_count,
            "total_tokens": self.total_usage.get("total_tokens", 0),
        }

    def __repr__(self) -> str:
        return (
            f"SessionSnapshot(id={self.session_id!r}, "
            f"turns={self.turn_count}, messages={self.message_count})"
        )


class PersistenceManager:
    """负责把 Session 状态持久化到磁盘，以及从磁盘还原。"""

    def __init__(self, base_dir: Path = Path(".ai_agent")):
        self.base_dir = Path(base_dir)
        self.sessions_dir = self.base_dir / "sessions"
        self.checkpoints_dir = self.base_dir / "checkpoints"
        self._ensure_dirs()

    def _ensure_dirs(self):
        self.sessions_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoints_dir.mkdir(parents=True, exist_ok=True)

    def _now_iso(self) -> str:
        return datetime.now(timezone.utc).isoformat()

    def _write_json_secure(self, path: Path, data: dict):
        """写入 JSON 文件，权限设为 600。"""
        path.parent.mkdir(parents=True, exist_ok=True)
        json_str = json.dumps(data, ensure_ascii=False, indent=2)
        path.write_text(json_str, encoding="utf-8")
        path.chmod(stat.S_IRUSR | stat.S_IWUSR)

    def _read_json(self, path: Path) -> Optional[dict]:
        if not path.exists():
            return None
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as e:
            print(f"读取文件失败 {path}: {e}")
            return None

    def _session_to_snapshot(self, session) -> SessionSnapshot:
        now = self._now_iso()
        session_id = getattr(session, "session_id", None) or str(uuid.uuid4())
        created_at = getattr(session, "created_at", now)
        turn_count = getattr(session, "turn_count", 0)

        ctx = getattr(session, "context_manager", None)
        if ctx is not None:
            messages = ctx.get_messages()
            system_prompt = getattr(ctx, "system_prompt", "")
        else:
            messages = []
            system_prompt = ""

        raw_usage = getattr(session, "total_usage", {})
        if hasattr(raw_usage, "__dict__"):
            total_usage = {
                "prompt_tokens": getattr(raw_usage, "prompt_tokens", 0),
                "completion_tokens": getattr(raw_usage, "completion_tokens", 0),
                "total_tokens": getattr(raw_usage, "total_tokens", 0),
            }
        elif isinstance(raw_usage, dict):
            total_usage = {
                "prompt_tokens": raw_usage.get("prompt_tokens", 0),
                "completion_tokens": raw_usage.get("completion_tokens", 0),
                "total_tokens": raw_usage.get("total_tokens", 0),
            }
        else:
            total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

        return SessionSnapshot(
            session_id=session_id,
            created_at=created_at,
            updated_at=now,
            turn_count=turn_count,
            messages=messages,
            total_usage=total_usage,
            system_prompt=system_prompt,
        )

    async def save_session(self, session) -> Path:
        """保存会话到 sessions/{session_id}.json（覆盖更新）。"""
        snapshot = self._session_to_snapshot(session)
        path = self.sessions_dir / f"{snapshot.session_id}.json"
        self._write_json_secure(path, snapshot.to_dict())
        return path

    async def load_session(self, session_id: str) -> Optional[SessionSnapshot]:
        """加载会话快照，不存在时返回 None。"""
        path = self.sessions_dir / f"{session_id}.json"
        data = self._read_json(path)
        if data is None:
            return None
        try:
            return SessionSnapshot.from_dict(data)
        except ValueError as e:
            print(f"加载会话 {session_id} 失败: {e}")
            return None

    def list_sessions(self) -> list:
        """返回所有已保存会话的摘要，按 updated_at 倒序排列。"""
        result = []
        for path in self.sessions_dir.glob("*.json"):
            data = self._read_json(path)
            if data is None:
                continue
            try:
                snap = SessionSnapshot.from_dict(data)
                result.append(snap.summary())
            except ValueError:
                continue
        result.sort(key=lambda x: x["updated_at"], reverse=True)
        return result

    async def save_checkpoint(self, session) -> str:
        """保存检查点，返回 checkpoint_id。"""
        snapshot = self._session_to_snapshot(session)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        checkpoint_id = f"{snapshot.session_id}_{ts}"
        path = self.checkpoints_dir / f"{checkpoint_id}.json"
        if path.exists():
            ts_ms = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:20]
            checkpoint_id = f"{snapshot.session_id}_{ts_ms}"
            path = self.checkpoints_dir / f"{checkpoint_id}.json"
        snapshot.metadata["checkpoint_id"] = checkpoint_id
        self._write_json_secure(path, snapshot.to_dict())
        return checkpoint_id

    async def load_checkpoint(self, checkpoint_id: str) -> Optional[SessionSnapshot]:
        """加载检查点快照。"""
        if checkpoint_id.endswith(".json"):
            checkpoint_id = checkpoint_id[:-5]
        path = self.checkpoints_dir / f"{checkpoint_id}.json"
        data = self._read_json(path)
        if data is None:
            return None
        try:
            return SessionSnapshot.from_dict(data)
        except ValueError as e:
            print(f"加载检查点 {checkpoint_id} 失败: {e}")
            return None

    def list_checkpoints(self, session_id: Optional[str] = None) -> list:
        """列出检查点。session_id=None 返回全部。"""
        result = []
        for path in self.checkpoints_dir.glob("*.json"):
            data = self._read_json(path)
            if data is None:
                continue
            try:
                snap = SessionSnapshot.from_dict(data)
                if session_id is not None and snap.session_id != session_id:
                    continue
                summary = snap.summary()
                summary["checkpoint_id"] = snap.metadata.get("checkpoint_id", path.stem)
                result.append(summary)
            except ValueError:
                continue
        result.sort(key=lambda x: x["updated_at"], reverse=True)
        return result

    async def restore_session(self, session, snapshot: SessionSnapshot):
        """把 snapshot 的状态恢复到 session 中。"""
        ctx = getattr(session, "context_manager", None)
        if ctx is None:
            raise AttributeError("session 没有 context_manager 属性")

        ctx.clear_messages()

        for msg in snapshot.messages:
            role = msg.get("role")
            content = msg.get("content", "")
            if role == "system":
                continue
            elif role == "user":
                ctx.add_user_message(content)
            elif role == "assistant":
                tool_calls = msg.get("tool_calls")
                ctx.add_assistant_message(content, tool_calls=tool_calls)
            elif role == "tool":
                tool_call_id = msg.get("tool_call_id", "")
                ctx.add_tool_result(tool_call_id, content)

        session.turn_count = snapshot.turn_count

        raw = snapshot.total_usage
        existing_usage = getattr(session, "total_usage", None)
        if hasattr(existing_usage, "prompt_tokens"):
            existing_usage.prompt_tokens = raw.get("prompt_tokens", 0)
            existing_usage.completion_tokens = raw.get("completion_tokens", 0)
            existing_usage.total_tokens = raw.get("total_tokens", 0)
        else:
            session.total_usage = dict(raw)


class PersistenceCLI:
    """把 PersistenceManager 功能封装为 /command 风格的处理器。"""

    def __init__(self, manager: PersistenceManager):
        self.manager = manager

    async def handle(self, command: str, session) -> bool:
        """
        处理持久化相关命令。
        返回 True 表示命令已处理，False 表示不是持久化命令。
        """
        parts = command.strip().split()
        if not parts or not parts[0].startswith("/"):
            return False

        cmd = parts[0].lower()
        args = parts[1:]

        if cmd == "/save":
            await self._cmd_save(session)
        elif cmd == "/sessions":
            await self._cmd_sessions()
        elif cmd == "/resume":
            await self._cmd_resume(args, session)
        elif cmd == "/checkpoint":
            await self._cmd_checkpoint(session)
        elif cmd == "/restore":
            await self._cmd_restore(args, session)
        else:
            return False

        return True

    async def _cmd_save(self, session):
        try:
            path = await self.manager.save_session(session)
            print(f"已保存会话 [{session.session_id}] -> {path}")
        except Exception as e:
            print(f"保存失败: {e}")

    async def _cmd_sessions(self):
        sessions = self.manager.list_sessions()
        if not sessions:
            print("没有已保存的会话")
            return
        print(f"{\'ID\':<20} {\'对话轮数\':>8} {\'消息数\':>6} {\'最后更新\':>20}")
        print("-" * 58)
        for s in sessions:
            updated = s["updated_at"][:19].replace("T", " ")
            print(f"{s[\'session_id\']:<20} {s[\'turn_count\']:>8} {s[\'message_count\']:>6} {updated:>20}")

    async def _cmd_resume(self, args: list, session):
        if not args:
            print("用法: /resume <session_id>")
            return
        session_id = args[0]
        snap = await self.manager.load_session(session_id)
        if snap is None:
            print(f"找不到会话: {session_id}")
            return
        try:
            await self.manager.restore_session(session, snap)
            print(f"已恢复会话 [{session_id}]，共 {snap.turn_count} 轮对话")
        except Exception as e:
            print(f"恢复失败: {e}")

    async def _cmd_checkpoint(self, session):
        try:
            ck_id = await self.manager.save_checkpoint(session)
            print(f"检查点已保存: {ck_id}")
        except Exception as e:
            print(f"保存检查点失败: {e}")

    async def _cmd_restore(self, args: list, session):
        if not args:
            checkpoints = self.manager.list_checkpoints(session_id=session.session_id)
            if not checkpoints:
                print(f"会话 [{session.session_id}] 没有检查点")
            else:
                print("可用检查点:")
                for ck in checkpoints:
                    print(f"  {ck[\'checkpoint_id\']} | turns={ck[\'turn_count\']}")
            return

        checkpoint_id = args[0]
        snap = await self.manager.load_checkpoint(checkpoint_id)
        if snap is None:
            print(f"找不到检查点: {checkpoint_id}")
            return
        try:
            await self.manager.restore_session(session, snap)
            print(f"已恢复到检查点 [{checkpoint_id}]，共 {snap.turn_count} 轮对话")
        except Exception as e:
            print(f"恢复失败: {e}")
'''

# 修复 f-string 中的转义问题（直接写文件时用正常 Python 语法）
persistence_final = '''"""
persistence.py — 会话持久化与检查点管理

两种持久化粒度：
  session:    保存整个对话历史，可以跨时间恢复继续
  checkpoint: 某个时间点的快照，可以回退到之前的状态

文件布局：
  .ai_agent/
  ├── sessions/{session_id}.json
  └── checkpoints/{session_id}_{YYYYMMDD_HHMMSS}.json
"""

import json
import stat
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional


@dataclass
class SessionSnapshot:
    """会话状态的完整快照，可序列化为 JSON。"""

    session_id: str
    created_at: str
    updated_at: str
    turn_count: int
    messages: list
    total_usage: dict
    system_prompt: str = ""
    metadata: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "session_id": self.session_id,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "turn_count": self.turn_count,
            "messages": self.messages,
            "total_usage": self.total_usage,
            "system_prompt": self.system_prompt,
            "metadata": self.metadata,
        }

    @classmethod
    def from_dict(cls, data: dict) -> "SessionSnapshot":
        required = {"session_id", "created_at", "updated_at", "turn_count", "messages", "total_usage"}
        missing = required - data.keys()
        if missing:
            raise ValueError(f"快照数据缺少字段: {missing}")
        usage = data["total_usage"]
        for key in ("prompt_tokens", "completion_tokens", "total_tokens"):
            usage.setdefault(key, 0)
        messages = []
        for msg in data["messages"]:
            msg = dict(msg)
            if "tool_calls" in msg and msg["tool_calls"] is not None:
                if not isinstance(msg["tool_calls"], list):
                    msg["tool_calls"] = []
            messages.append(msg)
        return cls(
            session_id=data["session_id"],
            created_at=data["created_at"],
            updated_at=data["updated_at"],
            turn_count=data["turn_count"],
            messages=messages,
            total_usage=usage,
            system_prompt=data.get("system_prompt", ""),
            metadata=data.get("metadata", {}),
        )

    @property
    def message_count(self) -> int:
        return sum(1 for m in self.messages if m.get("role") != "system")

    def summary(self) -> dict:
        return {
            "session_id": self.session_id,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "turn_count": self.turn_count,
            "message_count": self.message_count,
            "total_tokens": self.total_usage.get("total_tokens", 0),
        }

    def __repr__(self) -> str:
        return (
            f"SessionSnapshot(id={self.session_id!r}, "
            f"turns={self.turn_count}, messages={self.message_count})"
        )


class PersistenceManager:
    """负责把 Session 状态持久化到磁盘，以及从磁盘还原。"""

    def __init__(self, base_dir: Path = Path(".ai_agent")):
        self.base_dir = Path(base_dir)
        self.sessions_dir = self.base_dir / "sessions"
        self.checkpoints_dir = self.base_dir / "checkpoints"
        self._ensure_dirs()

    def _ensure_dirs(self):
        self.sessions_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoints_dir.mkdir(parents=True, exist_ok=True)

    def _now_iso(self) -> str:
        return datetime.now(timezone.utc).isoformat()

    def _write_json_secure(self, path: Path, data: dict):
        path.parent.mkdir(parents=True, exist_ok=True)
        json_str = json.dumps(data, ensure_ascii=False, indent=2)
        path.write_text(json_str, encoding="utf-8")
        path.chmod(stat.S_IRUSR | stat.S_IWUSR)

    def _read_json(self, path: Path) -> Optional[dict]:
        if not path.exists():
            return None
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as e:
            print(f"读取文件失败 {path}: {e}")
            return None

    def _session_to_snapshot(self, session) -> SessionSnapshot:
        now = self._now_iso()
        session_id = getattr(session, "session_id", None) or str(uuid.uuid4())
        created_at = getattr(session, "created_at", now)
        turn_count = getattr(session, "turn_count", 0)
        ctx = getattr(session, "context_manager", None)
        if ctx is not None:
            messages = ctx.get_messages()
            system_prompt = getattr(ctx, "system_prompt", "")
        else:
            messages = []
            system_prompt = ""
        raw_usage = getattr(session, "total_usage", {})
        if hasattr(raw_usage, "__dict__"):
            total_usage = {
                "prompt_tokens": getattr(raw_usage, "prompt_tokens", 0),
                "completion_tokens": getattr(raw_usage, "completion_tokens", 0),
                "total_tokens": getattr(raw_usage, "total_tokens", 0),
            }
        elif isinstance(raw_usage, dict):
            total_usage = {
                "prompt_tokens": raw_usage.get("prompt_tokens", 0),
                "completion_tokens": raw_usage.get("completion_tokens", 0),
                "total_tokens": raw_usage.get("total_tokens", 0),
            }
        else:
            total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
        return SessionSnapshot(
            session_id=session_id,
            created_at=created_at,
            updated_at=now,
            turn_count=turn_count,
            messages=messages,
            total_usage=total_usage,
            system_prompt=system_prompt,
        )

    async def save_session(self, session) -> Path:
        snapshot = self._session_to_snapshot(session)
        path = self.sessions_dir / f"{snapshot.session_id}.json"
        self._write_json_secure(path, snapshot.to_dict())
        return path

    async def load_session(self, session_id: str) -> Optional[SessionSnapshot]:
        path = self.sessions_dir / f"{session_id}.json"
        data = self._read_json(path)
        if data is None:
            return None
        try:
            return SessionSnapshot.from_dict(data)
        except ValueError as e:
            print(f"加载会话 {session_id} 失败: {e}")
            return None

    def list_sessions(self) -> list:
        result = []
        for path in self.sessions_dir.glob("*.json"):
            data = self._read_json(path)
            if data is None:
                continue
            try:
                snap = SessionSnapshot.from_dict(data)
                result.append(snap.summary())
            except ValueError:
                continue
        result.sort(key=lambda x: x["updated_at"], reverse=True)
        return result

    async def save_checkpoint(self, session) -> str:
        snapshot = self._session_to_snapshot(session)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        checkpoint_id = f"{snapshot.session_id}_{ts}"
        path = self.checkpoints_dir / f"{checkpoint_id}.json"
        if path.exists():
            ts_ms = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:20]
            checkpoint_id = f"{snapshot.session_id}_{ts_ms}"
            path = self.checkpoints_dir / f"{checkpoint_id}.json"
        snapshot.metadata["checkpoint_id"] = checkpoint_id
        self._write_json_secure(path, snapshot.to_dict())
        return checkpoint_id

    async def load_checkpoint(self, checkpoint_id: str) -> Optional[SessionSnapshot]:
        if checkpoint_id.endswith(".json"):
            checkpoint_id = checkpoint_id[:-5]
        path = self.checkpoints_dir / f"{checkpoint_id}.json"
        data = self._read_json(path)
        if data is None:
            return None
        try:
            return SessionSnapshot.from_dict(data)
        except ValueError as e:
            print(f"加载检查点 {checkpoint_id} 失败: {e}")
            return None

    def list_checkpoints(self, session_id: Optional[str] = None) -> list:
        result = []
        for path in self.checkpoints_dir.glob("*.json"):
            data = self._read_json(path)
            if data is None:
                continue
            try:
                snap = SessionSnapshot.from_dict(data)
                if session_id is not None and snap.session_id != session_id:
                    continue
                summary = snap.summary()
                summary["checkpoint_id"] = snap.metadata.get("checkpoint_id", path.stem)
                result.append(summary)
            except ValueError:
                continue
        result.sort(key=lambda x: x["updated_at"], reverse=True)
        return result

    async def restore_session(self, session, snapshot: SessionSnapshot):
        ctx = getattr(session, "context_manager", None)
        if ctx is None:
            raise AttributeError("session 没有 context_manager 属性")
        ctx.clear_messages()
        for msg in snapshot.messages:
            role = msg.get("role")
            content = msg.get("content", "")
            if role == "system":
                continue
            elif role == "user":
                ctx.add_user_message(content)
            elif role == "assistant":
                tool_calls = msg.get("tool_calls")
                ctx.add_assistant_message(content, tool_calls=tool_calls)
            elif role == "tool":
                tool_call_id = msg.get("tool_call_id", "")
                ctx.add_tool_result(tool_call_id, content)
        session.turn_count = snapshot.turn_count
        raw = snapshot.total_usage
        existing_usage = getattr(session, "total_usage", None)
        if hasattr(existing_usage, "prompt_tokens"):
            existing_usage.prompt_tokens = raw.get("prompt_tokens", 0)
            existing_usage.completion_tokens = raw.get("completion_tokens", 0)
            existing_usage.total_tokens = raw.get("total_tokens", 0)
        else:
            session.total_usage = dict(raw)


class PersistenceCLI:
    """把 PersistenceManager 功能封装为 /command 风格的处理器。"""

    def __init__(self, manager: PersistenceManager):
        self.manager = manager

    async def handle(self, command: str, session) -> bool:
        parts = command.strip().split()
        if not parts or not parts[0].startswith("/"):
            return False
        cmd = parts[0].lower()
        args = parts[1:]
        if cmd == "/save":
            await self._cmd_save(session)
        elif cmd == "/sessions":
            await self._cmd_sessions()
        elif cmd == "/resume":
            await self._cmd_resume(args, session)
        elif cmd == "/checkpoint":
            await self._cmd_checkpoint(session)
        elif cmd == "/restore":
            await self._cmd_restore(args, session)
        else:
            return False
        return True

    async def _cmd_save(self, session):
        try:
            path = await self.manager.save_session(session)
            print(f"已保存会话 [{session.session_id}] -> {path}")
        except Exception as e:
            print(f"保存失败: {e}")

    async def _cmd_sessions(self):
        sessions = self.manager.list_sessions()
        if not sessions:
            print("没有已保存的会话")
            return
        col1, col2, col3, col4 = "ID", "对话轮数", "消息数", "最后更新"
        print(f"{col1:<20} {col2:>8} {col3:>6} {col4:>20}")
        print("-" * 58)
        for s in sessions:
            updated = s["updated_at"][:19].replace("T", " ")
            print(f"{s[\'session_id\']:<20} {s[\'turn_count\']:>8} {s[\'message_count\']:>6} {updated:>20}")

    async def _cmd_resume(self, args: list, session):
        if not args:
            print("用法: /resume <session_id>")
            return
        snap = await self.manager.load_session(args[0])
        if snap is None:
            print(f"找不到会话: {args[0]}")
            return
        try:
            await self.manager.restore_session(session, snap)
            print(f"已恢复会话 [{args[0]}]，共 {snap.turn_count} 轮对话")
        except Exception as e:
            print(f"恢复失败: {e}")

    async def _cmd_checkpoint(self, session):
        try:
            ck_id = await self.manager.save_checkpoint(session)
            print(f"检查点已保存: {ck_id}")
        except Exception as e:
            print(f"保存检查点失败: {e}")

    async def _cmd_restore(self, args: list, session):
        if not args:
            checkpoints = self.manager.list_checkpoints(session_id=session.session_id)
            if not checkpoints:
                print(f"会话 [{session.session_id}] 没有检查点")
            else:
                print("可用检查点:")
                for ck in checkpoints:
                    print(f"  {ck[\'checkpoint_id\']} | turns={ck[\'turn_count\']}")
            return
        snap = await self.manager.load_checkpoint(args[0])
        if snap is None:
            print(f"找不到检查点: {args[0]}")
            return
        try:
            await self.manager.restore_session(session, snap)
            print(f"已恢复到检查点 [{args[0]}]，共 {snap.turn_count} 轮对话")
        except Exception as e:
            print(f"恢复失败: {e}")
'''

output_path = src_dir / "persistence.py"
output_path.write_text(persistence_final, encoding="utf-8")
print(f"已保存到: {output_path.resolve()}")
print(f"文件大小: {output_path.stat().st_size} 字节")

## 小结

| 模块 | 作用 | 存储格式 |
|------|------|----------|
| `SessionSnapshot` | 会话状态的可序列化快照 | JSON dict |
| `PersistenceManager.save_session` | 把当前会话写入磁盘（覆盖） | `sessions/{id}.json` |
| `PersistenceManager.load_session` | 从磁盘读取会话快照 | 权限 600 |
| `PersistenceManager.save_checkpoint` | 创建不可变的时间点快照 | `checkpoints/{id}_{ts}.json` |
| `PersistenceManager.restore_session` | 把快照状态回写到活跃 session | 内存操作 |
| `PersistenceCLI` | 把持久化操作封装为 /command | — |

关键设计决策：
- **文件权限 600**：对话内容可能含有敏感信息，只允许所有者读写
- **会话覆盖 vs 检查点追加**：会话文件随 `/save` 更新；检查点只追加，永不修改
- **tool_calls 防御**：反序列化时主动修复非 list 类型的 `tool_calls` 字段
- **与 Session 解耦**：`PersistenceManager` 通过 `getattr` 访问属性，不依赖具体类

下一章实现**多 Agent 协作**：多个 Agent 共享工具、分工处理子任务，并通过消息队列协调执行顺序。